# Question 1 – Predicting Maize Yield for Farmer Cooperatives in Rwanda

| Item | Details |
|---|---|
| **Group number** | Group XX *(replace)* |
| **Members** | Student 1 name *(replace)* and Student 2 name *(replace)* |
| **Question** | Q1 – Maize yield (MCSU scenario) |
| **Stages covered** | Stage 1 Environment · Stage 2 Acquire · Stage 3 EDA · Stage 4 Clean · Stage 5 Features · Stage 6 Finalise |
| **Dataset** | `maize_yield_rwanda_rw.csv` |
| **Target** | `yield_t_ha` (maize yield in tonnes per hectare) |

**Rules we follow in this notebook**

- The original file is never edited. It is copied to `data/raw/` and every change is made in code.
- No prediction model is built or trained. The work ends with a model-ready dataset.
- Every decision is written in the Cleaning Log (what, why, how many records).
- The test set is never used to take a decision. Imputing, scaling and feature selection are learned from the training set only.

---
# Stage 1 – Configure the environment (PC 1.1)

We import every library used in the project, print the versions, fix one random seed and create the project folders.

In [ ]:
# Standard library
import sys, platform, json, shutil, hashlib
from pathlib import Path

# Data and charts
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# scikit-learn: splitting, encoding, scaling, feature selection (NO model is trained)
import sklearn
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression, mutual_info_regression

SEED = 42                      # one fixed seed for the whole project
np.random.seed(SEED)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

versions = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "scikit-learn": sklearn.__version__,
    "joblib": joblib.__version__,
}
print("LIBRARY VERSIONS")
for name, v in versions.items():
    print(f"  {name:13s} {v}")
print("\nPython executable:", sys.executable)

In [ ]:
# Project folders (same structure as in the task sheet)
RAW_DIR       = Path("data/raw")         # original data, never edited
INTERIM_DIR   = Path("data/interim")     # data being cleaned
PROCESSED_DIR = Path("data/processed")   # final model-ready data
DOCS_DIR      = Path("docs")             # dictionary, logs, memo
for folder in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# environment.yml written from the versions that are really running,
# so that another person can rebuild the same environment
env_lines = [
    "name: ml-project",
    "channels:",
    "  - conda-forge",
    "dependencies:",
    f"  - python={versions['python']}",
    f"  - pandas={versions['pandas']}",
    f"  - numpy={versions['numpy']}",
    f"  - matplotlib={versions['matplotlib']}",
    f"  - seaborn={versions['seaborn']}",
    f"  - scikit-learn={versions['scikit-learn']}",
    f"  - joblib={versions['joblib']}",
    "  - notebook",
]
Path("environment.yml").write_text("\n".join(env_lines) + "\n", encoding="utf-8")
print(Path("environment.yml").read_text())
print("Folders ready:", [str(p) for p in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, DOCS_DIR]])

**Why these tools**

| Tool | Why we use it |
|---|---|
| Jupyter Notebook | Code, output and explanations stay together, so the work can be checked and repeated |
| pandas / numpy | Load the table, clean it and calculate new columns |
| matplotlib / seaborn | Charts for the data audit |
| scikit-learn | Train/validation/test split, encoding, scaling and feature selection with a fixed seed |
| joblib | Save the fitted pre-processing pipeline so it can be reused on new data |
| Git | Keep the history of the work and show the contribution of both members |

---
# Stage 2 – Define requirements and acquire data (PC 1.2)

## 2.1 Data requirements (from Mr. Habimana's message and the five business questions)

| Requirement | What we need |
|---|---|
| **Target** | A reliable maize yield per hectare for each farm record (Question A) |
| **Inputs** (Question B) | Seed variety and fertiliser, comparable between farms of different size (per hectare) |
| **Weather and location** (Question C) | Rainfall, temperature, altitude and irrigation, with valid values |
| **Place and time** (Question D) | One consistent label for each province, district and season; the period of the record |
| **Risk of low harvest** (Question E) | Only information known **before** harvest; no column that reveals the result |
| **Period** | Several years, all three seasons (A, B, C) |
| **Area** | Maize-growing provinces of Rwanda |
| **Size** | At least about 1,000 records so that a 70/15/15 split still leaves enough test records |
| **Privacy** | No personal information; record codes are allowed |

In [ ]:
# 2.2 Acquire: copy the supplied file into data/raw/ (the copy is never edited)
SOURCE_FILE = Path("maize_yield_rwanda_rw.csv")
RAW_FILE = RAW_DIR / SOURCE_FILE.name

if not RAW_FILE.exists():
    if not SOURCE_FILE.exists():
        raise FileNotFoundError(f"Put {SOURCE_FILE.name} next to this notebook or in {RAW_DIR}/")
    shutil.copy2(SOURCE_FILE, RAW_FILE)

def md5(path):
    """Fingerprint of a file: if the file changes, the fingerprint changes."""
    return hashlib.md5(Path(path).read_bytes()).hexdigest()

RAW_MD5 = md5(RAW_FILE)
print("Raw file :", RAW_FILE)
print("Size     :", RAW_FILE.stat().st_size, "bytes")
print("MD5      :", RAW_MD5, "(proof that the raw file is not changed)")

In [ ]:
# 2.3 Load the data and look at it
raw = pd.read_csv(RAW_FILE)
print("Rows:", raw.shape[0], "| Columns:", raw.shape[1])
display(raw.head())
display(raw.tail())

In [ ]:
raw.info()

## 2.4 Does the file meet the requirements?

The task sheet describes a file with columns such as `cooperative_id`, `planting_date`, `farm_size_unit`, `farmer_age`, `harvest_weight_kg` and `yield_kg_per_ha`. We compare that description with the file we really received.

In [ ]:
# Columns described in the task sheet
described = ["farm_id", "province", "district", "cooperative_id", "altitude_m", "season", "planting_date",
             "farm_size", "farm_size_unit", "seed_variety", "fertilizer_kg", "rainfall_mm", "avg_temp_c",
             "soil_type", "soil_ph", "pest_incidence", "irrigation", "farmer_age", "harvest_weight_kg",
             "yield_kg_per_ha"]
actual = list(raw.columns)

print("In the task sheet but NOT in the file:", sorted(set(described) - set(actual)))
print("In the file but NOT in the task sheet:", sorted(set(actual) - set(described)))
print()
print("Years in the file    :", raw["year"].min(), "to", raw["year"].max())
print("Provinces in the file:", raw["province"].nunique(), sorted(raw["province"].unique()))
print("Districts in the file:", raw["district"].nunique())
print("Seasons in the file  :", sorted(raw["season"].unique()))

requirement_check = pd.DataFrame([
    ["Target: yield per hectare",            "yield_t_ha",                                   "Met (unit is tonnes/ha, not kg/ha)"],
    ["B: fertiliser comparable per hectare", "npk_kg_ha, urea_kg_ha, manure_t_ha",           "Met (already per hectare)"],
    ["B: seed variety",                      "seed_variety",                                 "Met"],
    ["C: rainfall, temperature, altitude",   "rainfall_mm, avg_temp_c, altitude_m",          "Met (a few rainfall values missing)"],
    ["C: irrigation",                        "irrigation",                                   "Met"],
    ["D: province, district, season",        "province, district, season",                   "Met"],
    ["D: planting date",                     "year only",                                    "Partly met (no planting_date column)"],
    ["D: results by cooperative",            "none",                                         "NOT met (no cooperative_id column)"],
    ["E: only pre-harvest information",      "no harvest_weight column in the file",         "Met (leakage check in Stage 4)"],
    ["Privacy: no personal information",     "record_id is a record code; no farmer_age",    "Met"],
    ["Size: about 1,000 records or more",    f"{len(raw)} records",                          "Met"],
], columns=["Requirement", "Column(s) in the file", "Result"])
display(requirement_check)

**What we found.** The file we received is not the same as the one described in the task sheet. It has 1,426 records (not about 1,200), covers 2015–2024 (not 2023–2025), includes Kigali, and has no cooperative, planting-date, farmer-age or harvest-weight column. The yield is in tonnes per hectare and fertiliser is already per hectare.

**What we decided.** We work with the file as received and we do not add or invent columns. Questions A, B, C and E can be supported. Question D can be supported for provinces, districts and seasons, but **not for cooperatives**. This limit is reported in the memo.

In [ ]:
# 2.5 Data Dictionary (Appendix B). Meanings come from the column names and the task sheet;
#     the "observed" column is calculated from the file.
meanings = {
    "record_id":            ("Record code of the farm observation",                 "text",     "-",            "Identifier only, not a feature"),
    "year":                 ("Year of the season",                                   "integer",  "year",         "2015 to 2024"),
    "season":               ("Agricultural season",                                  "category", "-",            "A, B, C"),
    "province":             ("Province of the farm",                                 "category", "-",            "5 provinces"),
    "district":             ("District of the farm",                                 "category", "-",            "30 districts"),
    "altitude_m":           ("Altitude of the farm",                                 "integer",  "metres",       "Positive"),
    "rainfall_mm":          ("Rainfall during the season",                           "decimal",  "mm",           "Positive; has missing values"),
    "avg_temp_c":           ("Average temperature during the season",                "decimal",  "deg C",        "Maize grows best at about 18 to 27"),
    "soil_ph":              ("Soil acidity",                                         "decimal",  "pH",           "Farmland about 4 to 8; has missing values"),
    "soil_type":            ("Type of soil",                                         "category", "-",            "5 types"),
    "seed_variety":         ("Maize seed variety planted",                           "category", "-",            "Local or a named improved variety"),
    "farming_practice":     ("Farming practice recorded by the officer",             "category", "-",            "Improved, Traditional (no definition supplied)"),
    "npk_kg_ha":            ("NPK fertiliser applied per hectare",                   "decimal",  "kg/ha",        "0 = none applied"),
    "urea_kg_ha":           ("Urea fertiliser applied per hectare",                  "decimal",  "kg/ha",        "0 = none applied"),
    "manure_t_ha":          ("Manure applied per hectare",                           "decimal",  "t/ha",         "0 = none applied"),
    "plant_density_per_ha": ("Number of plants per hectare",                         "integer",  "plants/ha",    "Positive"),
    "farm_area_ha":         ("Size of the farm",                                     "decimal",  "hectares",     "Positive; one unit only"),
    "irrigation":           ("Whether the plot was irrigated",                       "category", "-",            "Yes, No"),
    "pest_incidence_index": ("Level of pests as an index (assumed 0 = none, 1 = severe)", "decimal", "index",   "0 to 1; has missing values"),
    "yield_t_ha":           ("Maize yield per hectare - THE TARGET",                 "decimal",  "t/ha",         "1 t/ha = 1,000 kg/ha"),
}

def observed(col):
    s = raw[col]
    if s.dtype == "object":
        return f"{s.nunique()} unique values" if s.nunique() > 8 else ", ".join(sorted(s.dropna().unique()))
    return f"{s.min()} to {s.max()}"

data_dictionary = pd.DataFrame(
    [[c, *meanings[c], str(raw[c].dtype), observed(c), int(raw[c].isna().sum())] for c in raw.columns],
    columns=["Column name", "Meaning", "Data type", "Unit", "Allowed values / notes",
             "pandas dtype", "Observed values", "Missing"])
data_dictionary.to_csv(DOCS_DIR / "data_dictionary.csv", index=False)
display(data_dictionary)

In [ ]:
# 2.6 Source Log
source_log = pd.DataFrame([
    ["Source",             "File supplied with the practical task (Question 1, MCSU scenario)"],
    ["File name",          RAW_FILE.name],
    ["Link",               "Not applicable - file given by the lecturer"],
    ["Date accessed",      pd.Timestamp.today().strftime("%d %B %Y")],
    ["Licence",            "For this course only (task sheet, Section 9)"],
    ["Who collected it",   "According to the scenario: MCSU field officers; the task sheet says the data is synthetic"],
    ["Size",               f"{raw.shape[0]} rows x {raw.shape[1]} columns"],
    ["MD5 of raw file",    RAW_MD5],
    ["Limitations",        "Does not match the description in the task sheet: no cooperative, planting date, farmer age or harvest weight; years 2015-2024"],
    ["Ethical issues",     "No personal information in the file. Bias risk: Season C and irrigated plots have few records"],
], columns=["Field", "Entry"])
source_log.to_csv(DOCS_DIR / "source_log.csv", index=False)
display(source_log)

---
# Stage 3 – Explore and audit the data (PC 1.3)

In this stage we only **look**. Nothing is changed. Every problem found goes into the Data Quality Report at the end of the stage.

## 3.1 Description of the dataset

In [ ]:
num_cols = raw.select_dtypes("number").columns.tolist()
cat_cols = [c for c in raw.select_dtypes("object").columns if c != "record_id"]

print("Shape:", raw.shape)
print("Numerical columns  :", num_cols)
print("Categorical columns:", cat_cols)
display(raw[num_cols].describe().T.round(2))

In [ ]:
# Unique values of every categorical column (to see spelling or format problems)
for c in cat_cols:
    counts = raw[c].value_counts(dropna=False)
    print(f"\n{c}  ({raw[c].nunique()} unique values)")
    print(counts.to_string())

**What we found.** Every category has one spelling only (no `kigali` / `Kigali`, no extra spaces). Seasons A and B have 600 records each and Season C has 226. Only 128 plots (about 9%) are irrigated.

## 3.2 Missing values

In [ ]:
missing = pd.DataFrame({
    "Missing_Count": raw.isnull().sum(),
    "Missing_Percentage": (raw.isnull().mean() * 100).round(2),
}).sort_values("Missing_Count", ascending=False)
display(missing[missing["Missing_Count"] > 0])

rows_with_missing = int(raw.isnull().any(axis=1).sum())
print("Records with at least one missing value:", rows_with_missing, "of", len(raw))

# Disguised missing values: text such as "?", "NA", "unknown", or codes such as -999 / 999
disguised_text = {c: int(raw[c].astype(str).str.strip().str.lower().isin(["", "?", "na", "n/a", "none", "null", "unknown", "-"]).sum())
                  for c in raw.select_dtypes("object").columns}
disguised_num = {c: int(raw[c].isin([-999, -99, -1, 999, 9999]).sum()) for c in num_cols}
print("Disguised missing (text):", sum(disguised_text.values()))
print("Disguised missing (number codes):", sum(disguised_num.values()))

# Chart 1 - where the missing values are
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(raw.isnull(), cbar=False, yticklabels=False, cmap=["#f0f0f0", "#d9534f"], ax=ax)
ax.set_title("Chart 1: Missing values in the maize dataset (red = missing)")
ax.set_xlabel("Columns")
plt.tight_layout(); plt.show()

**Chart 1 – interpretation.** Only three columns have missing values: `rainfall_mm` (11), `soil_ph` (14) and `pest_incidence_index` (14). They sit on the same 14 records, so these look like incomplete forms. This is less than 1% of the file, so we keep the records and fill the gaps (Stage 4) instead of deleting them. No disguised missing values were found.

## 3.3 Duplicates

In [ ]:
dup_full   = int(raw.duplicated().sum())
dup_id     = int(raw["record_id"].duplicated().sum())
dup_no_id  = int(raw.drop(columns="record_id").duplicated().sum())
print("Complete duplicate rows                   :", dup_full)
print("Repeated record_id                        :", dup_id)
print("Same values with a different record_id    :", dup_no_id)

**What we found.** There are no repeated records, even when the record code is ignored. Nothing has to be removed.

## 3.4 Impossible and unrealistic values

We use the field notes from the MCSU agronomist. *Impossible* means the value cannot exist (for example a negative yield). *Outside the typical range* means the value must be checked, but it is not automatically wrong.

In [ ]:
# (column, lowest possible, highest possible, typical low, typical high, source of the rule)
RULES = [
    ("year",                 2015, 2024,   2015, 2024,  "Period of the file"),
    ("altitude_m",            900, 3000,    900, 3000,  "Farmland altitude in Rwanda (our assumption)"),
    ("rainfall_mm",             0, 2500,      0, 2500,  "Rain cannot be negative"),
    ("avg_temp_c",              5,   35,     18,   27,  "Agronomist: maize grows best at 18-27 C"),
    ("soil_ph",                 0,   14,      4,    8,  "Agronomist: farmland pH about 4-8"),
    ("npk_kg_ha",               0, 1000,      0, 1000,  "Amount cannot be negative"),
    ("urea_kg_ha",              0, 1000,      0, 1000,  "Amount cannot be negative"),
    ("manure_t_ha",             0,  100,      0,  100,  "Amount cannot be negative"),
    ("plant_density_per_ha",    1, 150000,    1, 150000, "Must be positive"),
    ("farm_area_ha",        0.001,  100,  0.001,  100,  "Must be positive"),
    ("pest_incidence_index",    0,    1,      0,    1,  "Index between 0 and 1 (our assumption)"),
    ("yield_t_ha",          0.001,   15,      1,    4,  "Agronomist: typical 1,000-4,000 kg/ha"),
]
range_check = []
for col, lo, hi, tlo, thi, why in RULES:
    s = raw[col]
    impossible = int(((s < lo) | (s > hi)).sum())
    untypical  = int(((s < tlo) | (s > thi)).sum()) - impossible
    range_check.append([col, s.min(), s.max(), f"{lo} to {hi}", impossible, f"{tlo} to {thi}", untypical, why])
range_check = pd.DataFrame(range_check, columns=["Column", "Min in file", "Max in file", "Possible range",
                                                 "Impossible values", "Typical range", "Outside typical range", "Rule"])
display(range_check)

n_temp_low  = int((raw["avg_temp_c"] < 18).sum())
n_yield_hi  = int((raw["yield_t_ha"] > 4).sum())
print("Records cooler than 18 C:", n_temp_low, "- mean altitude of these records:",
      round(raw.loc[raw["avg_temp_c"] < 18, "altitude_m"].mean()), "m (all records:", round(raw["altitude_m"].mean()), "m)")
print("Records with yield above 4 t/ha:", n_yield_hi)
print(raw.loc[raw["yield_t_ha"] > 4, "seed_variety"].value_counts().to_string())

In [ ]:
# Consistency checks between columns
district_two_provinces = int((raw.groupby("district")["province"].nunique() > 1).sum())
print("Districts recorded in more than one province:", district_two_provinces)

print("\nIrrigation by season:")
display(pd.crosstab(raw["season"], raw["irrigation"], margins=True))

rain_floor_value = raw["rainfall_mm"].min()
rain_floor_n = int((raw["rainfall_mm"] == rain_floor_value).sum())
print(f"Records with rainfall exactly at the minimum ({rain_floor_value} mm):", rain_floor_n)
print(raw.loc[raw["rainfall_mm"] == rain_floor_value, "season"].value_counts().to_string())

local_improved = int(((raw["seed_variety"] == "Local") & (raw["farming_practice"] == "Improved")).sum())
print("\nRecords with Local seed but 'Improved' farming practice:", local_improved)

zero_npk, zero_urea, zero_manure = [int((raw[c] == 0).sum()) for c in ["npk_kg_ha", "urea_kg_ha", "manure_t_ha"]]
print("Records with zero NPK / urea / manure:", zero_npk, "/", zero_urea, "/", zero_manure)

**What we found.**

- No impossible value exists in the file (no negative amount, no pH outside 0–14, no yield of zero or less).
- Many records are cooler than 18 °C. They are at higher altitude than the average farm, which agrees with the agronomist's note about the cool highlands. We keep them.
- A few yields are above 4 t/ha. They belong to named improved varieties, which the agronomist says can give more. We keep them.
- Several records have rainfall exactly equal to the lowest value in the file, all in Season C. This looks like a lower limit applied when the weather data was added. The values are possible for the dry season, so we keep them and report the point as a limit.
- Fertiliser and manure have many zeros. Zero means "none applied", not "missing", so we keep the zeros.
- Some records combine `Local` seed with `Improved` farming practice. `farming_practice` therefore does not mean "improved seed"; we keep both columns and create a clear seed flag in Stage 5.

## 3.5 Outliers (box plots and the IQR rule)

In [ ]:
def iqr_outliers(s):
    """Number of values further than 1.5 x IQR from the quartiles."""
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < low) | (s > high)).sum()), round(low, 2), round(high, 2)

outlier_cols = [c for c in num_cols if c != "year"]
outlier_table = pd.DataFrame(
    [[c, *iqr_outliers(raw[c].dropna()), round(raw[c].skew(), 2)] for c in outlier_cols],
    columns=["Column", "IQR outliers", "Lower limit", "Upper limit", "Skewness"]).sort_values("IQR outliers", ascending=False)
display(outlier_table)

# Chart 2 - box plots
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, c in zip(axes.ravel(), outlier_cols):
    sns.boxplot(x=raw[c], ax=ax, color="#5b9bd5")
    ax.set_title(c)
    ax.set_xlabel("")
for ax in axes.ravel()[len(outlier_cols):]:
    ax.set_visible(False)
fig.suptitle("Chart 2: Box plots of the numerical columns", fontsize=14)
plt.tight_layout(); plt.show()

**Chart 2 – interpretation.** `farm_area_ha` has the most outliers and is strongly skewed to the right: most farms are below 1 ha and a few are up to about 4 ha. These are realistic larger farms, not errors, so we keep them and use a log transformation in Stage 5. The few outliers in `soil_ph`, `pest_incidence_index`, `manure_t_ha` and `yield_t_ha` are all inside the possible range, so they are kept.

## 3.6 The target: `yield_t_ha`

In [ ]:
# Chart 3 - distribution of the target
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(raw["yield_t_ha"], bins=30, kde=True, color="#70ad47", ax=ax)
ax.axvline(raw["yield_t_ha"].mean(), color="black", linestyle="--", label=f"Mean = {raw['yield_t_ha'].mean():.2f} t/ha")
ax.axvspan(1, 4, color="grey", alpha=0.10, label="Typical range 1-4 t/ha")
ax.set_title("Chart 3: Distribution of maize yield (target)")
ax.set_xlabel("Yield (t/ha)"); ax.set_ylabel("Number of records"); ax.legend()
plt.tight_layout(); plt.show()

print(raw["yield_t_ha"].describe().round(2).to_string())
print("Skewness:", round(raw["yield_t_ha"].skew(), 2))

**Chart 3 – interpretation.** The yield is close to a bell shape, centred at about 2.6 t/ha, with almost no skew. Nearly all values are inside the typical range of 1 to 4 t/ha. The target is a continuous number, so this is a **regression** problem and the target needs no transformation.

## 3.7 Relationships with the target

In [ ]:
# Chart 4 - yield by season and by seed variety (Questions B and D)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.barplot(data=raw, x="season", y="yield_t_ha", order=["A", "B", "C"], color="#5b9bd5", ax=axes[0])
axes[0].set_title("Mean yield by season"); axes[0].set_ylabel("Mean yield (t/ha)")
order = raw.groupby("seed_variety")["yield_t_ha"].mean().sort_values().index
sns.barplot(data=raw, x="seed_variety", y="yield_t_ha", order=order, color="#ed7d31", ax=axes[1])
axes[1].set_title("Mean yield by seed variety"); axes[1].set_ylabel("Mean yield (t/ha)")
fig.suptitle("Chart 4: Mean yield by season and by seed variety", fontsize=14)
plt.tight_layout(); plt.show()

print(raw.groupby("season")["yield_t_ha"].mean().round(2).to_string())
print(raw.groupby("seed_variety")["yield_t_ha"].mean().round(2).sort_values().to_string())

**Chart 4 – interpretation.** Season A gives the highest mean yield and Season C the lowest. Local seed gives the lowest mean yield of all varieties; every named variety is higher. Season and seed variety are therefore useful features for Questions B and D.

In [ ]:
# Chart 5 - rainfall against yield, coloured by season (Question C)
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=raw, x="rainfall_mm", y="yield_t_ha", hue="season", hue_order=["A", "B", "C"], alpha=0.6, ax=ax)
ax.set_title("Chart 5: Rainfall and yield, by season")
ax.set_xlabel("Rainfall in the season (mm)"); ax.set_ylabel("Yield (t/ha)")
plt.tight_layout(); plt.show()
print(raw.groupby("season")["rainfall_mm"].describe()[["count", "mean", "min", "max"]].round(0).to_string())

**Chart 5 – interpretation.** Yield rises with rainfall. Season C forms a separate group with much less rain and lower yield, so rainfall depends strongly on the season. For this reason missing rainfall values will be filled with the median **of the same season**, not with one overall value.

In [ ]:
# Chart 6 - correlation heatmap of the numerical columns
corr = raw[num_cols].corr()
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax)
ax.set_title("Chart 6: Correlation between numerical columns")
plt.tight_layout(); plt.show()
print("Correlation with the target:")
print(corr["yield_t_ha"].drop("yield_t_ha").sort_values(ascending=False).round(2).to_string())

**Chart 6 – interpretation.** Rainfall and plant density have the strongest positive link with yield, followed by urea and NPK; the pest index has a negative link. Altitude and temperature are strongly and negatively linked with each other (high farms are cooler), which confirms the agronomist's note. No single column is linked to the target strongly enough to look like a hidden copy of the result.

In [ ]:
# Chart 7 - yield by province and by irrigation (Questions C and D)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.boxplot(data=raw, x="province", y="yield_t_ha", color="#a5a5a5", ax=axes[0])
axes[0].set_title("Yield by province"); axes[0].set_ylabel("Yield (t/ha)")
sns.boxplot(data=raw, x="season", y="yield_t_ha", hue="irrigation", order=["A", "B", "C"], ax=axes[1])
axes[1].set_title("Yield by season and irrigation"); axes[1].set_ylabel("Yield (t/ha)")
fig.suptitle("Chart 7: Yield by province, and by season and irrigation", fontsize=14)
plt.tight_layout(); plt.show()
print(raw.groupby("province")["yield_t_ha"].agg(["count", "mean"]).round(2).to_string())
print(raw.groupby(["season", "irrigation"])["yield_t_ha"].agg(["count", "mean"]).round(2).to_string())

**Chart 7 – interpretation.** The table printed under the chart gives the mean yield of each province and of irrigated and non-irrigated plots in each season. The effect of irrigation must be read **inside each season**, because irrigated plots are mostly in the dry Season C; comparing all irrigated plots with all non-irrigated plots would mix the season effect with the irrigation effect. This is why we create a rainfall × irrigation feature in Stage 5.

## 3.8 Data Quality Report

In [ ]:
n_farm_out = int(outlier_table.set_index("Column").loc["farm_area_ha", "IQR outliers"])
quality_report = pd.DataFrame([
    ["File differs from the description in the task sheet", "cooperative_id, planting_date, farmer_age, harvest_weight_kg absent",
        len(raw), "High", "Work with the file as received; report that results by cooperative are not possible"],
    ["Missing values", "rainfall_mm, soil_ph, pest_incidence_index",
        rows_with_missing, "Medium", "Keep the records; fill with the median learned from the training set (rainfall: median of the same season)"],
    ["Few records for Season C", "season",
        int((raw["season"] == "C").sum()), "Medium", "Keep; stratify the split by season so every set has Season C"],
    ["Few irrigated plots (imbalance)", "irrigation",
        int((raw["irrigation"] == "Yes").sum()), "Medium", "Keep; report that irrigation results are less certain"],
    ["Rainfall values at a lower limit", "rainfall_mm",
        rain_floor_n, "Low", "Keep (possible in the dry season); report as a limit"],
    ["Skewed column with outliers", "farm_area_ha",
        n_farm_out, "Low", "Keep (realistic farms); apply a log transformation"],
    ["Other IQR outliers inside the possible range", "soil_ph, pest_incidence_index, manure_t_ha, yield_t_ha",
        int(outlier_table.set_index("Column").loc[["soil_ph", "pest_incidence_index", "manure_t_ha", "yield_t_ha"], "IQR outliers"].sum()),
        "Low", "Keep; values are realistic"],
    ["Values outside the typical range", "avg_temp_c below 18 C; yield_t_ha above 4 t/ha",
        n_temp_low + n_yield_hi, "Low", "Checked: highland farms and improved varieties; keep"],
    ["Zeros that could be confused with missing", "npk_kg_ha, urea_kg_ha, manure_t_ha",
        int(((raw["npk_kg_ha"] == 0) | (raw["urea_kg_ha"] == 0) | (raw["manure_t_ha"] == 0)).sum()),
        "Low", "Keep as real zeros (none applied); add a fertiliser-used flag"],
    ["Unclear category meaning", "farming_practice against seed_variety",
        local_improved, "Low", "Keep both; create a clear improved_seed flag"],
    ["Identifier column", "record_id",
        len(raw), "Medium", "Do not use as a feature (it would let a model memorise records)"],
    ["Text stored as Yes/No", "irrigation",
        len(raw), "Low", "Convert to 1/0"],
    ["Duplicates", "all columns", dup_full + dup_no_id, "None", "Checked: none found"],
    ["Inconsistent spelling or labels", "all text columns", 0, "None", "Checked: none found"],
    ["Impossible values", "all numerical columns", int(range_check["Impossible values"].sum()), "None", "Checked: none found"],
    ["Leakage (column known only after harvest)", "all columns", 0, "None", "Checked: only the target is a harvest result"],
], columns=["Problem", "Columns affected", "Records affected", "Severity", "Proposed treatment"])
quality_report.to_csv(DOCS_DIR / "data_quality_report.csv", index=False)
display(quality_report)

**Note on the "eight hidden problems".** The task sheet says that at least eight kinds of problems are hidden in the described file (duplicates, mixed units, mixed spellings, and so on). We ran each of these checks on the file we received and we report the true result: several of them find nothing. We did not invent problems that the data does not show.

---
# Stage 4 – Clean and pre-process the data (PC 1.3)

Each step is a small function, so that the same steps can be repeated on new data. Each step shows a before and after view and is written into the Cleaning Log.

In [ ]:
cleaning_log = []

def log_step(step, columns, action, reason, records_affected, before, after):
    """Add one line to the Cleaning Log."""
    cleaning_log.append({"Step": len(cleaning_log) + 1, "Operation": step, "Columns": columns, "Action": action,
                         "Reason": reason, "Records affected": int(records_affected), "Before": str(before), "After": str(after)})
    print(f"[logged] {step}: {records_affected} record(s) affected")

df = raw.copy()          # 'raw' stays untouched
print("Start of cleaning:", df.shape)

In [ ]:
# STEP 1 - Tidy text: remove extra spaces so that each category has one label
TEXT_COLS = ["record_id", "season", "province", "district", "soil_type", "seed_variety", "farming_practice", "irrigation"]

def tidy_text(d):
    d = d.copy()
    for c in TEXT_COLS:
        d[c] = d[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    d["season"] = d["season"].str.upper()
    return d

before_unique = {c: df[c].nunique() for c in TEXT_COLS if c != "record_id"}
after = tidy_text(df)
changed = int((after[TEXT_COLS] != df[TEXT_COLS].astype(str)).any(axis=1).sum())
df = after
after_unique = {c: df[c].nunique() for c in TEXT_COLS if c != "record_id"}
print("Unique values before:", before_unique)
print("Unique values after :", after_unique)
log_step("Tidy text labels", ", ".join(TEXT_COLS), "Strip spaces, season in capital letters",
         "One label for each category, season and district", changed, before_unique, after_unique)

In [ ]:
# STEP 2 - Duplicates: remove repeated farm records (same values, with or without the same record code)
def remove_duplicates(d):
    value_cols = [c for c in d.columns if c != "record_id"]
    return d.drop_duplicates(subset=value_cols, keep="first").drop_duplicates(subset="record_id", keep="first")

before_shape = df.shape
df = remove_duplicates(df)
print("Shape before:", before_shape, "| Shape after:", df.shape)
log_step("Remove duplicates", "all", "Keep the first of any repeated record",
         "No repeated farm records may remain", before_shape[0] - df.shape[0], before_shape, df.shape)

In [ ]:
# STEP 3 - Data types: irrigation Yes/No becomes 1/0; numbers are numbers
NUMERIC_COLS = ["year", "altitude_m", "rainfall_mm", "avg_temp_c", "soil_ph", "npk_kg_ha", "urea_kg_ha",
                "manure_t_ha", "plant_density_per_ha", "farm_area_ha", "pest_incidence_index", "yield_t_ha"]

def fix_types(d):
    d = d.copy()
    for c in NUMERIC_COLS:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")      # text that is not a number becomes missing
    d["irrigation"] = d["irrigation"].str.lower().map({"yes": 1, "no": 0})
    return d

before_types = df[["irrigation"]].dtypes.astype(str).to_dict()
before_counts = df["irrigation"].value_counts().to_dict()
missing_before = int(df[NUMERIC_COLS].isna().sum().sum())
df = fix_types(df)
print("irrigation before:", before_counts, before_types)
print("irrigation after :", df["irrigation"].value_counts().to_dict(), {"irrigation": str(df["irrigation"].dtype)})
print("New missing values created by the type conversion:", int(df[NUMERIC_COLS].isna().sum().sum()) - missing_before,
      "| irrigation not Yes/No:", int(df["irrigation"].isna().sum()))
log_step("Correct data types", "irrigation", "Yes -> 1, No -> 0",
         "A model needs numbers; the meaning is unchanged", len(df), before_counts, df["irrigation"].value_counts().to_dict())

In [ ]:
# STEP 4 - Impossible values become missing (they are then filled like other missing values)
POSSIBLE = {col: (lo, hi) for col, lo, hi, *_ in RULES}

def blank_impossible(d):
    d = d.copy()
    for col, (lo, hi) in POSSIBLE.items():
        if col in d.columns:
            d.loc[(d[col] < lo) | (d[col] > hi), col] = np.nan
    return d

missing_before = df.isna().sum()
df = blank_impossible(df)
n_blanked = int((df.isna().sum() - missing_before).sum())
print("Values set to missing because they are impossible:", n_blanked)
log_step("Impossible values", ", ".join(POSSIBLE), "Set to missing if outside the possible range",
         "An impossible value is an error and must not be trusted", n_blanked,
         f"{int(missing_before.sum())} missing cells", f"{int(df.isna().sum().sum())} missing cells")

In [ ]:
# STEP 5 - Outliers and unusual values: checked in Stage 3, kept on purpose (decision recorded)
log_step("Outliers: farm_area_ha", "farm_area_ha", "Keep; log transformation in Stage 5",
         "Larger farms are realistic, not errors", 0, f"{n_farm_out} IQR outliers", f"{n_farm_out} kept")
log_step("Outliers: other columns", "soil_ph, pest_incidence_index, manure_t_ha, yield_t_ha", "Keep",
         "All inside the possible range given by the agronomist", 0, "inside possible range", "kept")
log_step("Values outside typical range", "avg_temp_c, yield_t_ha", "Keep",
         "Cool records are highland farms; high yields are improved varieties", 0, f"{n_temp_low + n_yield_hi} checked", "kept")
log_step("Rainfall at lower limit", "rainfall_mm", "Keep; report as a limit",
         "Possible in the dry Season C", 0, f"{rain_floor_n} records at {rain_floor_value} mm", "kept")
log_step("Zeros in fertiliser and manure", "npk_kg_ha, urea_kg_ha, manure_t_ha", "Keep as zero",
         "Zero means none applied, not missing", 0, "zeros", "kept")

In [ ]:
# STEP 6 - Leakage check and target check
# Leakage = a column that is only known AFTER harvest and would give the answer away.
after_harvest_words = ["harvest", "yield", "production", "output", "sold", "income", "revenue"]
suspects = [c for c in df.columns if any(w in c.lower() for w in after_harvest_words) and c != "yield_t_ha"]
print("Columns that are only known after harvest (other than the target):", suspects)

TARGET = "yield_t_ha"
n_no_target = int(df[TARGET].isna().sum())
df = df.dropna(subset=[TARGET])          # a record without a target cannot be used for learning
print("Records without a target removed:", n_no_target)

log_step("Leakage check", "all", f"Remove columns known only after harvest: {suspects if suspects else 'none found'}",
         "Question E needs only information known before harvest", 0, df.shape[1], df.shape[1] - len(suspects))
df = df.drop(columns=suspects)
log_step("Records without target", TARGET, "Remove", "Cannot be used to learn or to test", n_no_target, len(df) + n_no_target, len(df))

In [ ]:
# One function that repeats steps 1 to 4 on any new raw file (reusable pipeline, part 1)
def clean_data(raw_df):
    """Tidy text -> remove duplicates -> correct types -> blank impossible values."""
    return blank_impossible(fix_types(remove_duplicates(tidy_text(raw_df))))

# Proof that the function gives the same result as the step-by-step cells above
assert clean_data(raw).equals(df), "clean_data() must reproduce the cleaned table"

df.to_csv(INTERIM_DIR / "maize_clean.csv", index=False)
print("Clean dataset saved:", INTERIM_DIR / "maize_clean.csv", df.shape)
print("\nMissing values still in the clean file (filled after the split, from the training set only):")
print(df.isna().sum()[df.isna().sum() > 0].to_string())

## 4.2 Split into training, validation and test sets (70 / 15 / 15)

The split is done **before** filling missing values and scaling. If we filled or scaled first, numbers from the validation and test records would influence the training data; that is data leakage.

- Fixed seed: `SEED = 42`, so the split is always the same.
- Stratified by season, so that every set has the same share of Seasons A, B and C (Season C is small).

In [ ]:
train, temp = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["season"])
val, test   = train_test_split(temp, test_size=0.50, random_state=SEED, stratify=temp["season"])
train, val, test = [d.reset_index(drop=True) for d in (train, val, test)]

print("Train     :", train.shape)
print("Validation:", val.shape)
print("Test      :", test.shape)
display(pd.DataFrame({"train": train["season"].value_counts(normalize=True),
                      "validation": val["season"].value_counts(normalize=True),
                      "test": test["season"].value_counts(normalize=True)}).round(3))

# No record may be in two sets
ids = [set(d["record_id"]) for d in (train, val, test)]
assert not (ids[0] & ids[1]) and not (ids[0] & ids[2]) and not (ids[1] & ids[2])
print("No record appears in more than one set.")
log_step("Split 70/15/15", "all", f"Stratified by season, random seed {SEED}", "Honest evaluation later; no leakage",
         len(df), len(df), f"train {len(train)}, validation {len(val)}, test {len(test)}")

## 4.3 Missing values (learned from the training set only)

| Column | Treatment | Reason |
|---|---|---|
| `rainfall_mm` | Median of the **same season** in the training set | Rainfall is very different between seasons (Chart 5) |
| `soil_ph` | Median of the training set | Few missing values; the median is not pulled by outliers |
| `pest_incidence_index` | Median of the training set | Same reason; the column is skewed, so median is better than mean |
| Any other numerical column | Median of the training set | Safety rule for new data |
| Any categorical column | Most frequent value of the training set | Safety rule for new data |

In [ ]:
CAT_COLS = ["season", "province", "district", "soil_type", "seed_variety", "farming_practice"]
NUM_FEATURES = [c for c in NUMERIC_COLS if c != TARGET] + ["irrigation"]

def fit_imputer(train_df):
    """Learn the fill values from the TRAINING set only."""
    return {
        "rain_by_season": train_df.groupby("season")["rainfall_mm"].median().to_dict(),
        "num_median": train_df[NUM_FEATURES].median().to_dict(),
        "cat_mode": {c: train_df[c].mode()[0] for c in CAT_COLS},
    }

def apply_imputer(d, values):
    """Fill missing values with the values learned by fit_imputer."""
    d = d.copy()
    d["rainfall_mm"] = d["rainfall_mm"].fillna(d["season"].map(values["rain_by_season"]))
    for c in NUM_FEATURES:
        d[c] = d[c].fillna(values["num_median"][c])
    for c in CAT_COLS:
        d[c] = d[c].fillna(values["cat_mode"][c])
    return d

impute_values = fit_imputer(train)
print("Rainfall median by season (training set):", {k: round(v, 1) for k, v in impute_values["rain_by_season"].items()})
print("soil_ph median (training set):", impute_values["num_median"]["soil_ph"])
print("pest_incidence_index median (training set):", impute_values["num_median"]["pest_incidence_index"])

before = {name: int(d.isna().sum().sum()) for name, d in [("train", train), ("validation", val), ("test", test)]}
train, val, test = [apply_imputer(d, impute_values) for d in (train, val, test)]
after = {name: int(d.isna().sum().sum()) for name, d in [("train", train), ("validation", val), ("test", test)]}
print("\nMissing cells before:", before)
print("Missing cells after :", after)
log_step("Fill missing values", "rainfall_mm, soil_ph, pest_incidence_index",
         "Rainfall: training median of the same season; others: training median",
         "Keep all records; values learned from the training set only", rows_with_missing, before, after)

# Save the clean splits (not yet scaled, still readable)
for name, d in [("train", train), ("validation", val), ("test", test)]:
    d.to_csv(INTERIM_DIR / f"{name}_clean.csv", index=False)
print("\nClean splits saved in", INTERIM_DIR)

## 4.4 Encoding and scaling

- **One-hot encoding** for categories without an order (season, province, soil type, seed variety, farming practice): each label becomes its own 0/1 column, so no false order is created.
- **Standardisation** (mean 0, standard deviation 1) for continuous numbers, because their units are very different (plants per hectare are in tens of thousands, pH is about 6).
- 0/1 columns are left as they are. The target is not scaled.
- The encoder and the scaler are **fitted on the training set only** and then applied to the validation and test sets.

In [ ]:
def build_preprocessor(scale_cols, passthrough_cols, onehot_cols):
    """scikit-learn ColumnTransformer: scale numbers, keep 0/1 columns, one-hot encode categories."""
    pre = ColumnTransformer([
        ("scale",  StandardScaler(), scale_cols),
        ("keep",   "passthrough", passthrough_cols),
        ("onehot", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False), onehot_cols),
    ], verbose_feature_names_out=False)
    return pre.set_output(transform="pandas")

base_scale  = [c for c in NUMERIC_COLS if c != TARGET]
base_onehot = ["season", "province", "soil_type", "seed_variety", "farming_practice"]
demo_pre = build_preprocessor(base_scale, ["irrigation"], base_onehot)
demo_train = demo_pre.fit_transform(train)          # fit on TRAIN only
demo_val   = demo_pre.transform(val)                # only transform

print("Columns before encoding:", len(base_scale) + 1 + len(base_onehot), "| after encoding:", demo_train.shape[1])
display(demo_train.head())
print("Scaled columns in the training set - mean and standard deviation:")
display(demo_train[base_scale].agg(["mean", "std"]).round(3))
print("Same columns in the validation set (not exactly 0 and 1, because the scaler was learned on the training set):")
display(demo_val[base_scale].agg(["mean", "std"]).round(3))
log_step("Encoding and scaling", ", ".join(base_onehot) + " / numerical columns", "One-hot encoding; standardisation",
         "Categories have no order; numbers have very different units. Fitted on the training set only", len(train),
         f"{len(base_scale) + 1 + len(base_onehot)} columns", f"{demo_train.shape[1]} columns")

In [ ]:
# 4.5 Target balance for Question E (farms at risk of a very low harvest)
# "Very low" = lowest 20% of yields. The limit is taken from the TRAINING set only.
LOW_YIELD_LIMIT = float(train[TARGET].quantile(0.20))
for d in (train, val, test):
    d["low_yield_risk"] = (d[TARGET] <= LOW_YIELD_LIMIT).astype(int)

print(f"Low-yield limit (20th percentile of the training set): {LOW_YIELD_LIMIT:.2f} t/ha")
display(pd.DataFrame({"train": train["low_yield_risk"].value_counts(normalize=True),
                      "validation": val["low_yield_risk"].value_counts(normalize=True),
                      "test": test["low_yield_risk"].value_counts(normalize=True)}).round(3))
log_step("Low-yield label for Question E", "low_yield_risk", f"1 if yield <= {LOW_YIELD_LIMIT:.2f} t/ha (training 20th percentile)",
         "Second target for Question E; it is NOT a feature. Imbalance about 20/80 - planned remedy: class weights, or resampling on training data only",
         len(df), "not present", "added to all three sets")

cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv(DOCS_DIR / "cleaning_log.csv", index=False)
display(cleaning_log_df)

**About `low_yield_risk`.** This column is made from the target, so it is a second **target** for Question E, never a feature. It is imbalanced (about 20% "at risk"). The remedy planned for the modelling team is class weights, or resampling applied to the training data only.

---
# Stage 5 – Engineer and select features (PC 1.4)

## 5.1 New features

All new features are calculated row by row from information known **before** harvest. The limits used for the zones are fixed numbers, so the same calculation is applied to the training, validation and test sets.

In [ ]:
def add_features(d):
    """Create the engineered features. Uses only pre-harvest columns."""
    d = d.copy()
    # Question B - inputs
    d["total_fert_kg_ha"] = d["npk_kg_ha"] + d["urea_kg_ha"]                    # all mineral fertiliser per hectare
    d["any_fertiliser"]   = (d["total_fert_kg_ha"] > 0).astype(int)             # 1 = some mineral fertiliser used
    d["improved_seed"]    = (d["seed_variety"] != "Local").astype(int)          # 1 = named improved variety
    d["manure_and_fert"]  = ((d["manure_t_ha"] > 0) & (d["total_fert_kg_ha"] > 0)).astype(int)  # both together
    # Question C - weather and location
    d["altitude_zone"]    = pd.cut(d["altitude_m"], [-np.inf, 1500, 1900, np.inf], labels=[0, 1, 2]).astype(int)  # 0 low, 1 middle, 2 high
    d["temp_optimal"]     = d["avg_temp_c"].between(18, 27).astype(int)         # agronomist: best at 18-27 C
    d["ph_optimal"]       = d["soil_ph"].between(5.5, 7.0).astype(int)          # agronomist: maize prefers pH 5.5-7
    d["rain_x_irrigation"] = d["rainfall_mm"] * d["irrigation"]                 # rainfall on irrigated plots (0 if not irrigated)
    # Question E - risk
    d["pest_level"]       = pd.cut(d["pest_incidence_index"], [-np.inf, 0.2, 0.4, np.inf], labels=[0, 1, 2]).astype(int)  # 0 low, 1 medium, 2 high
    # Skewed column
    d["log_farm_area"]    = np.log1p(d["farm_area_ha"])
    return d

NEW_FEATURES = ["total_fert_kg_ha", "any_fertiliser", "improved_seed", "manure_and_fert", "altitude_zone",
                "temp_optimal", "ph_optimal", "rain_x_irrigation", "pest_level", "log_farm_area"]

train_fe, val_fe, test_fe = [add_features(d) for d in (train, val, test)]
print("Columns before:", train.shape[1], "| after:", train_fe.shape[1])
display(train_fe[["record_id"] + NEW_FEATURES].head(8))

**Assumptions.** The altitude limits (1,500 m and 1,900 m) and the pest limits (0.2 and 0.4) are round numbers chosen by us to make three groups; no official limits were supplied with the file. `improved_seed` treats every named variety as improved, because the file does not say which varieties are hybrids.

## 5.2 Skewness and transformation

In [ ]:
skew_cols = [c for c in NUMERIC_COLS if c not in (TARGET, "year")]
skew_table = train_fe[skew_cols].skew().round(2).sort_values(ascending=False).to_frame("Skewness (training set)")
display(skew_table)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train_fe["farm_area_ha"], bins=30, ax=axes[0], color="#5b9bd5")
axes[0].set_title(f"farm_area_ha (skewness {train_fe['farm_area_ha'].skew():.2f})")
sns.histplot(train_fe["log_farm_area"], bins=30, ax=axes[1], color="#70ad47")
axes[1].set_title(f"log_farm_area (skewness {train_fe['log_farm_area'].skew():.2f})")
fig.suptitle("Chart 8: Farm area before and after the log transformation", fontsize=14)
plt.tight_layout(); plt.show()

**Chart 8 – interpretation.** `farm_area_ha` is the only strongly skewed column (skewness well above 1). The log transformation pulls the few large farms closer to the others and reduces the skewness. The other columns are close to symmetric and are left as they are.

## 5.3 Relationship of every feature with the target (training set only)

In [ ]:
candidate_numeric = [c for c in NUMERIC_COLS if c != TARGET] + ["irrigation"] + NEW_FEATURES
y_train = train_fe[TARGET]

corr_target = train_fe[candidate_numeric].corrwith(y_train)
mi = pd.Series(mutual_info_regression(train_fe[candidate_numeric], y_train, random_state=SEED), index=candidate_numeric)
relation = pd.DataFrame({"Correlation with yield": corr_target.round(3), "Mutual information": mi.round(3),
                         "New feature": [c in NEW_FEATURES for c in candidate_numeric]})
relation = relation.reindex(relation["Correlation with yield"].abs().sort_values(ascending=False).index)
display(relation)

fig, ax = plt.subplots(figsize=(9, 7))
colors = ["#ed7d31" if c in NEW_FEATURES else "#5b9bd5" for c in relation.index]
ax.barh(relation.index[::-1], relation["Correlation with yield"][::-1], color=colors[::-1])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Chart 9: Correlation of each feature with yield (orange = new feature)")
ax.set_xlabel("Correlation with yield_t_ha (training set)")
plt.tight_layout(); plt.show()

**Chart 9 – interpretation.** The table and chart rank all numerical features by the strength of their link with yield, on the training set only. Correlation measures straight-line links; mutual information also catches links that are not straight lines. We use mutual information instead of a Random Forest importance because our manager asked us not to train a model.

## 5.4 Feature selection

Four steps, all decided on the **training set only**:

1. Remove constant features (the same value for every record).
2. Remove duplicated features (two columns with identical values).
3. Remove one feature of each highly correlated pair (|correlation| above 0.90), keeping the one more strongly linked with yield. The same check is repeated after one-hot encoding.
4. `SelectKBest` (score: F-test for regression) on the encoded and scaled features, keeping the columns whose link with yield is statistically significant (p-value below 0.05).
5. Business rule: columns that the five business questions need (irrigation, altitude, temperature, province) are kept even if their score is low.

`district` is not used as a model feature: it has 30 labels with about 33 training records each, and every district lies inside one province, which is kept. `district` remains in the clean files in `data/interim/` for reporting by district (Question D). `record_id` is never a feature.

In [ ]:
selection_notes = {}     # feature -> reason for removal

# Step 1 - constant features
constant = [c for c in candidate_numeric if train_fe[c].nunique() <= 1]
for c in constant: selection_notes[c] = "Removed: constant in the training set"
print("1. Constant features:", constant)

# Step 2 - duplicated features
remaining = [c for c in candidate_numeric if c not in constant]
dup_mask = train_fe[remaining].T.duplicated()
duplicated = dup_mask[dup_mask].index.tolist()
for c in duplicated: selection_notes[c] = "Removed: identical to another feature"
print("2. Duplicated features:", duplicated)

# Step 3 - highly correlated pairs
remaining = [c for c in remaining if c not in duplicated]
corr_matrix = train_fe[remaining].corr().abs()
high_pairs, dropped_corr = [], []
for i, a in enumerate(remaining):
    for b in remaining[i + 1:]:
        if corr_matrix.loc[a, b] > 0.90:
            high_pairs.append((a, b, round(corr_matrix.loc[a, b], 3)))
for a, b, r in sorted(high_pairs, key=lambda p: -p[2]):
    if a in dropped_corr or b in dropped_corr:
        continue
    weaker = a if abs(corr_target[a]) < abs(corr_target[b]) else b
    stronger = b if weaker == a else a
    dropped_corr.append(weaker)
    selection_notes[weaker] = f"Removed: correlation {r} with {stronger}, which is more strongly linked with yield"
print("3. Highly correlated pairs (|r| > 0.90):")
for p in high_pairs: print("   ", p)
print("   Removed:", dropped_corr)

kept_numeric = [c for c in remaining if c not in dropped_corr]
print("\nNumerical features kept after steps 1-3:", len(kept_numeric))
print(kept_numeric)

In [ ]:
# Final pre-processor: fitted on the training set only
BINARY_ORDINAL = ["irrigation", "any_fertiliser", "improved_seed", "manure_and_fert", "altitude_zone",
                  "temp_optimal", "ph_optimal", "pest_level"]
ONEHOT_COLS = ["season", "province", "soil_type", "seed_variety", "farming_practice"]

scale_cols = [c for c in kept_numeric if c not in BINARY_ORDINAL]       # continuous numbers -> standardised
pass_cols  = [c for c in kept_numeric if c in BINARY_ORDINAL]           # 0/1 and 0/1/2 columns -> kept as they are

preprocessor = build_preprocessor(scale_cols, pass_cols, ONEHOT_COLS)
X_train_all = preprocessor.fit_transform(train_fe)      # FIT on train
X_val_all   = preprocessor.transform(val_fe)
X_test_all  = preprocessor.transform(test_fe)
print("Encoded and scaled feature matrix:", X_train_all.shape)

# Step 4a - variance threshold (removes any column that is constant after encoding)
vt = VarianceThreshold(threshold=0.0).fit(X_train_all)
after_vt = X_train_all.columns[vt.get_support()].tolist()
print("4a. Removed by the variance threshold:", sorted(set(X_train_all.columns) - set(after_vt)))

# Step 4b - the high-correlation check again, now that categories are 0/1 columns
enc_corr = X_train_all[after_vt].corr().abs()
redundant = []
for i, a in enumerate(after_vt):
    for b in after_vt[i + 1:]:
        if enc_corr.loc[a, b] > 0.90 and a not in redundant and b not in redundant:
            redundant.append(b)          # keep the first column, remove the later one
            print(f"4b. {b} removed: correlation {enc_corr.loc[a, b]:.2f} with {a}")
candidates = [c for c in after_vt if c not in redundant]

# Step 4c - SelectKBest with the F-test for regression
# K is not guessed: it is the number of columns whose link with yield is statistically significant (p < 0.05)
_, p_values = f_regression(X_train_all[candidates], y_train)
K_BEST = int((p_values < 0.05).sum())
print(f"4c. Columns with a significant link with yield (p < 0.05): K = {K_BEST}")
skb = SelectKBest(score_func=f_regression, k=K_BEST).fit(X_train_all[candidates], y_train)
kbest = [c for c, keep in zip(candidates, skb.get_support()) if keep]

# Step 4d - business rule: columns needed by Questions C and D stay in the dataset
BUSINESS_KEEP = [c for c in candidates if c in ("irrigation", "altitude_m", "avg_temp_c") or c.startswith("province_")]
SELECTED_FEATURES = [c for c in candidates if c in kbest or c in BUSINESS_KEEP]

scores = pd.DataFrame({"F score": skb.scores_.round(2), "p-value": skb.pvalues_.round(4)}, index=candidates)
scores["Decision"] = ["Kept: SelectKBest" if c in kbest else "Kept: business rule" if c in BUSINESS_KEEP else "Removed"
                      for c in candidates]
display(scores.sort_values("F score", ascending=False))

print(f"4c. SelectKBest kept {len(kbest)} of {len(candidates)} columns.")
print(f"4d. Kept by the business rule although the score is low: {[c for c in BUSINESS_KEEP if c not in kbest]}")
print(f"\nFINAL LIST OF SELECTED FEATURES ({len(SELECTED_FEATURES)}):")
for i, c in enumerate(SELECTED_FEATURES, 1):
    print(f"  {i:2d}. {c}")

**What we decided.**

- `SelectKBest` keeps the K columns with the highest F score. We did not guess K: it is the number of columns whose link with yield is statistically significant on the training set (p-value below 0.05). The removed columns have a link so weak that it could be chance.
- The F-test looks at one column at a time, so it can miss an effect that is hidden by another column. Irrigation is the clear example: irrigated plots are mostly in the dry Season C, so over the whole file they do not look better, but **inside every season** irrigated plots have a higher mean yield (table under Chart 7). Removing irrigation would be a mistake.
- For this reason irrigation, altitude, temperature and province are kept by a business rule: Questions C and D cannot be answered without them. The modelling team can change `K_BEST` or `BUSINESS_KEEP` and re-run the notebook.

## 5.5 Feature Justification table

In [ ]:
def status(feature):
    """Final status of a numerical feature after selection."""
    if feature in selection_notes:
        return selection_notes[feature]
    if feature in kbest:
        return "Kept"
    if feature in SELECTED_FEATURES:
        return "Kept (business rule: needed for Questions C and D)"
    return "Removed by SelectKBest: weak link with yield"

feature_info = {
    "total_fert_kg_ha":  ("npk_kg_ha + urea_kg_ha", "B", "All mineral fertiliser per hectare, comparable between farms of any size"),
    "any_fertiliser":    ("1 if total_fert_kg_ha > 0", "B", "Separates farms using no mineral fertiliser from the others"),
    "improved_seed":     ("1 if seed_variety is not Local", "B", "Clear answer to: does improved seed increase yield?"),
    "manure_and_fert":   ("1 if manure > 0 and mineral fertiliser > 0", "B", "Farms combining organic and mineral inputs"),
    "altitude_zone":     ("altitude_m in 3 groups: below 1500, 1500-1900, above 1900 m", "C, D", "Lowland / middle / highland comparison"),
    "temp_optimal":      ("1 if avg_temp_c is between 18 and 27", "C", "Agronomist: maize grows best in this range"),
    "ph_optimal":        ("1 if soil_ph is between 5.5 and 7", "C", "Agronomist: maize prefers this soil acidity"),
    "rain_x_irrigation": ("rainfall_mm x irrigation", "C", "Lets the effect of rain differ on irrigated plots"),
    "pest_level":        ("pest_incidence_index in 3 groups: up to 0.2, 0.2-0.4, above 0.4", "E", "Simple Low / Medium / High pest risk for early support"),
    "log_farm_area":     ("log(1 + farm_area_ha)", "A", "Reduces the strong right skew of farm size"),
}
feature_table = pd.DataFrame(
    [[f, how, q, why, round(corr_target[f], 3), round(mi[f], 3), status(f)] for f, (how, q, why) in feature_info.items()],
    columns=["Feature", "How it was created", "Business question", "Purpose", "Correlation with yield (train)",
             "Mutual information (train)", "Decision"])
feature_table.to_csv(DOCS_DIR / "feature_justification.csv", index=False)
display(feature_table)

original_decisions = pd.DataFrame(
    [[c, round(corr_target[c], 3), status(c)] for c in candidate_numeric if c not in NEW_FEATURES]
    + [["district", np.nan, "Not used as a feature: 30 labels, each inside one province (kept in data/interim)"],
       ["record_id", np.nan, "Not used as a feature: identifier"]],
    columns=["Original column", "Correlation with yield (train)", "Decision"])
original_decisions.to_csv(DOCS_DIR / "original_feature_decisions.csv", index=False)
display(original_decisions)

---
# Stage 6 – Finalise, package and present (PC 1.1 – 1.4)

## 6.1 Save the model-ready datasets and the pipeline

In [ ]:
def final_table(X_all, d):
    """Selected features + the target + the Question E label."""
    out = X_all[SELECTED_FEATURES].copy()
    out[TARGET] = d[TARGET].values
    out["low_yield_risk"] = d["low_yield_risk"].values
    return out.round(6)

final_train, final_val, final_test = final_table(X_train_all, train_fe), final_table(X_val_all, val_fe), final_table(X_test_all, test_fe)
final_train.to_csv(PROCESSED_DIR / "train.csv", index=False)
final_val.to_csv(PROCESSED_DIR / "validation.csv", index=False)
final_test.to_csv(PROCESSED_DIR / "test.csv", index=False)

# The fitted pipeline, saved so it can be reused on new data
pipeline_bundle = {"impute_values": impute_values, "preprocessor": preprocessor, "selected_features": SELECTED_FEATURES,
                   "low_yield_limit": LOW_YIELD_LIMIT, "target": TARGET, "seed": SEED}
joblib.dump(pipeline_bundle, PROCESSED_DIR / "preprocessing_pipeline.joblib")

def prepare_new_data(raw_df, bundle=pipeline_bundle):
    """REUSABLE PIPELINE: raw records -> model-ready features, using only what was learned from the training set."""
    d = clean_data(raw_df)
    d = apply_imputer(d, bundle["impute_values"])
    d = add_features(d)
    return bundle["preprocessor"].transform(d)[bundle["selected_features"]]

print("Pipeline test on 5 raw records:")
display(prepare_new_data(raw.sample(5, random_state=SEED)))

print("Files in", PROCESSED_DIR)
for p in sorted(PROCESSED_DIR.iterdir()):
    print(f"  {p.name:32s} {p.stat().st_size:>9,d} bytes")

In [ ]:
# 6.2 Final dataset information
final_train.info()
display(final_train.head())

In [ ]:
# 6.3 Definition of done - every check is calculated, not typed by hand
final_sets = {"train": final_train, "validation": final_val, "test": final_test}
checks = [
    ("Duplicates",       "No repeated farm records remain",
        df.drop(columns="record_id").duplicated().sum() == 0 and df["record_id"].is_unique),
    ("Data types",       "Every final column is a number",
        all(pd.api.types.is_numeric_dtype(t) for d in final_sets.values() for t in d.dtypes)),
    ("Consistency",      "One label per category; one unit for farm size (hectares)",
        all(df[c].str.strip().eq(df[c]).all() for c in CAT_COLS)),
    ("Realistic values", "No impossible value remains",
        all(df[c].dropna().between(lo, hi).all() for c, (lo, hi) in POSSIBLE.items())),
    ("Missing values",   "No missing value in the final files",
        all(d.isna().sum().sum() == 0 for d in final_sets.values())),
    ("No leakage",       "No after-harvest column and no identifier among the features",
        not any(w in c.lower() for c in SELECTED_FEATURES for w in ["yield", "harvest", "record_id"])),
    ("Split",            f"Three files saved, seed {SEED}, no record in two sets",
        len(final_train) + len(final_val) + len(final_test) == len(df)),
    ("Same columns",     "Train, validation and test have the same columns in the same order",
        list(final_train.columns) == list(final_val.columns) == list(final_test.columns)),
    ("Features",         "At least five new features created and explained",
        len(NEW_FEATURES) >= 5),
    ("Raw file",         "The raw file is unchanged since the start",
        md5(RAW_FILE) == RAW_MD5),
]
done = pd.DataFrame([[a, b, "PASS" if bool(ok) else "FAIL"] for a, b, ok in checks], columns=["Check", "Standard", "Result"])
display(done)
assert (done["Result"] == "PASS").all(), "A definition-of-done check failed"

print("Fingerprints of the final files (the same on every run = repeatable):")
for name in ["train.csv", "validation.csv", "test.csv"]:
    print(f"  {name:16s} {md5(PROCESSED_DIR / name)}")

In [ ]:
# 6.4 Data Card (one page) - written from the values calculated above
n_new_kept = int(feature_table["Decision"].str.startswith("Kept").sum())
data_card = f"""# Data Card - Maize Yield, Rwanda (Question 1)

**Purpose.** Model-ready data to predict maize yield per hectare before harvest for the Maize Cooperative Support Unit (MCSU).

**Source.** `{RAW_FILE.name}`, supplied with the practical task. The task sheet states that the data is synthetic. For this course only.

**Size.** Raw file: {raw.shape[0]} records, {raw.shape[1]} columns. Usable after cleaning: {len(df)} records.
Split (seed {SEED}, stratified by season): train {len(final_train)}, validation {len(final_val)}, test {len(final_test)}.

**Targets.** `{TARGET}` (tonnes per hectare, regression). `low_yield_risk` (1 if yield is at or below {LOW_YIELD_LIMIT:.2f} t/ha, the 20th percentile of the training set) for Question E. `low_yield_risk` is a target, not a feature.

**Columns in the final files ({len(SELECTED_FEATURES)} features).** {", ".join(SELECTED_FEATURES)}.

**Cleaning steps.** Text labels tidied; duplicates checked (none); irrigation changed to 1/0; impossible values checked (none); outliers checked and kept with reasons; leakage check (no after-harvest column); missing values in rainfall, soil pH and pest index filled with training-set medians (rainfall by season).

**Feature steps.** {len(NEW_FEATURES)} new features created ({n_new_kept} kept after selection); log transformation of farm area; one-hot encoding; standardisation; selection by constant/duplicate/high-correlation filters and SelectKBest (F-test), all learned on the training set; irrigation, altitude, temperature and province kept by a business rule for Questions C and D.

**Known limitations and bias risks.**
- The file does not match the description in the task sheet: no cooperative, planting date, farmer age or harvest weight.
- Season C ({int((raw['season'] == 'C').sum())} records) and irrigated plots ({int((raw['irrigation'] == 'Yes').sum())} records) are few, so results for them are less certain.
- {rain_floor_n} rainfall values sit exactly at {rain_floor_value} mm, which suggests a lower limit in the weather data.
- Altitude and pest groups use limits chosen by the analysts.
- `district` is not a model feature; it stays in `data/interim/` for reporting.

**How to reproduce.** Create the environment from `environment.yml`, put the raw file next to the notebook (or in `data/raw/`), then use Kernel > Restart & Run All. The fingerprints printed in section 6.3 must be the same on every run.
"""
(DOCS_DIR / "data_card.md").write_text(data_card, encoding="utf-8")
display(Markdown(data_card))

In [ ]:
# 6.5 One-page memo to the manager (Appendix E) - numbers come from the notebook
season_means = raw.groupby("season")["yield_t_ha"].mean().round(2)
local_mean = raw.loc[raw["seed_variety"] == "Local", "yield_t_ha"].mean()
improved_mean = raw.loc[raw["seed_variety"] != "Local", "yield_t_ha"].mean()
kept_new = feature_table.loc[feature_table["Decision"].str.startswith("Kept"), "Feature"].tolist()

memo = f"""# MEMO

**To:** Mr. Habimana, Head of Planning and Data, MCSU
**From:** Student 1 name and Student 2 name, Junior Data Analysts (Group XX)
**Date:** {pd.Timestamp.today().strftime("%d %B %Y")}
**Subject:** Maize yield data: ready for modelling?

**1. What we received.** The file had {raw.shape[0]:,} farm records and {raw.shape[1]} columns, covering {raw['year'].min()} to {raw['year'].max()}, {raw['province'].nunique()} provinces, {raw['district'].nunique()} districts and the three seasons. After cleaning, all {len(df):,} records can be used.

**2. Main problems found.**
- *The file is not the one that was described to us.* It has no cooperative code, no planting date and no harvest weight. This matters because we cannot report results for each cooperative. We worked with the file as it is and did not invent the missing columns.
- *Some forms were incomplete.* {rows_with_missing} records had no rainfall, soil acidity or pest value. Leaving them empty would stop the forecast for those farms. We filled them with the middle value of similar records (for rainfall, records of the same season).
- *Season C and irrigated plots have few records* ({int((raw['season'] == 'C').sum())} Season C records, {int((raw['irrigation'] == 'Yes').sum())} irrigated plots). Forecasts for them will be less reliable. We made sure they are present in the same share in every part of the data.

**3. Data removed or changed.** No record was removed. No duplicate, impossible value or mixed spelling was found. Unusual values (large farms, cool highland farms, yields above 4 t/ha) were checked and kept because they are realistic. The record code and the district name are not given to the model. We found no column that is known only after harvest, so nothing had to be removed to avoid leakage; the yield itself is kept only as the value to predict.

**4. New features.** We created {len(NEW_FEATURES)} new pieces of information, for example total fertiliser per hectare, an improved-seed marker, altitude zone, a marker for good soil acidity and temperature, pest level, and rainfall on irrigated plots. After checking which ones add information, {len(kept_new)} are kept in the final data ({", ".join(kept_new)}); the others repeated information already present or showed a weak link with yield. They help with Questions B (inputs), C (weather and location) and E (risk). In the raw data, improved varieties average {improved_mean:.2f} t/ha against {local_mean:.2f} t/ha for local seed, and Season A averages {season_means['A']:.2f} t/ha against {season_means['C']:.2f} t/ha in Season C. These are simple averages, not yet proof of cause.

**5. Limits and risks.** The data cannot tell us anything about individual cooperatives, planting dates or farmers. {rain_floor_n} rainfall values sit exactly at {rain_floor_value:.0f} mm, which suggests the weather data has a lower limit. The task sheet states that the data is synthetic, so conclusions must be confirmed with real field data.

**6. Recommendation.** The data is ready for the modelling team: {len(final_train)} records for training, {len(final_val)} for validation and {len(final_test)} for the final test, with {len(SELECTED_FEATURES)} features. It can support Questions A, B, C and E now, and Question D for provinces and seasons. Next season, please ask field officers to record the cooperative code, the planting date and the actual rainfall for every farm, and to visit more Season C and irrigated plots.
"""
(DOCS_DIR / "memo_to_manager.md").write_text(memo, encoding="utf-8")
display(Markdown(memo))

In [ ]:
# 6.6 Everything produced by this notebook
print("PROJECT FILES")
for folder in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, DOCS_DIR]:
    print(f"\n{folder}/")
    for p in sorted(folder.iterdir()):
        if p.is_file():
            print(f"   {p.name:36s} {p.stat().st_size:>9,d} bytes")
print("\nNotebook finished without errors.")

## Before you submit

1. Replace **Group XX** and the two student names in the first cell and in the memo cell (section 6.5).
2. Use **Kernel > Restart & Run All** and take the screenshots listed in Section 2 of the task sheet.
3. Run it a second time and check that the fingerprints in section 6.3 are the same.
4. Commit the notebook, `environment.yml`, `data/` and `docs/` to the group repository (both members commit).